## Link Prediction with the Simple API

Link prediction Graph Neural Networks (GNNs) can predict the probability for an
edge to existe between two nodes.

**Part 1:** This tutorial shows how to train, analyze, and evaluate a link
prediction model on the Arxiv citation graph. The objective is to predict the
probability of citation between two papers i.e. train a model "Model(paper1,
paper2) -> probability of paper1 citying paper2".

**Part 2:** Running a model on each pair of papers/nodes is computationally
expensive. To improve serving scalability, we can use a decomposable model,
i.e., a model decomposable as "Model(paper1, paper2) := Encoder1(paper1) *
Encoder2(paper2)". By separating the model into two independent encoders, paper
representations can be pre-computed, and only the dot product is required for
the final prediction. We show this in this tutorial.

**Part 3:** If the goal is to retrieve the most likely citation for a specific
paper, calculating the probability of all possible other paper is still
inefficient. Instead, the output of one encoder can be indexed in a vector
database. This allows us for efficient retrieval of the *closest* paper. We show
this in this tutorial.

**[Not yet available] Part 4:** By default, GNNs use existing edges as both
features for message passing and as labels for the training objective. However,
some workflows require isolating specific edge sets so the model cannot "see"
them during the forward pass. We show how to handle cases where an edgeset acts
as a pure label.

**[Not yet available] Part 5:** Finally, production models must respect temporal
constraints, ensuring a model does not access future data to predict past
events. Training a model with this restriction, known as temporal-aware
modeling, ensures performance metrics remain realistic at serving time. The last
section of this tutorial demonstrates how to implement this temporal masking.

## Importing libraries

In [26]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import dgf  # Import Graph Flow
import jax
import numpy as np

## Get Graph data

We first fetch the Arxiv graph.

In [2]:
# Download the Mag graph from the OGB repo.
graph, schema = dgf.io.fetch_ogb_graph("arxiv")

Caching arxiv graph at /tmp/gf_fetch/arxiv.cache
OGB dependency not available. Downloading graph from CNS.


Let's look at the graph stucture.

In [3]:
# Show the schema
dgf.analyse.print_schema(schema)

Graph Schema:

Node Sets:
  nodes:
    | Feature   | Format     | Semantic    | Shape   | Num cat. vals   |
    |-----------|------------|-------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID  | None    | None            |
    | #split    | BYTES      | CATEGORICAL | None    | None            |
    | feat      | FLOAT_32   | EMBEDDING   | (128,)  | None            |
    | labels    | INTEGER_64 | CATEGORICAL | None    | 40              |
    | year      | INTEGER_64 | NUMERICAL   | None    | None            |


Edge Sets:
  edges: (Source: nodes, Target: nodes)
    (No features)



This graph has a "label" feature for node prediction. We don't need it for link
prediction, so we can mask it in the schema.

In [4]:
del schema.node_sets["nodes"].features["#split"]
del schema.node_sets["nodes"].features["labels"]
dgf.analyse.print_schema(schema)

Graph Schema:

Node Sets:
  nodes:
    | Feature   | Format     | Semantic   | Shape   | Num cat. vals   |
    |-----------|------------|------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID | None    | None            |
    | feat      | FLOAT_32   | EMBEDDING  | (128,)  | None            |
    | year      | INTEGER_64 | NUMERICAL  | None    | None            |


Edge Sets:
  edges: (Source: nodes, Target: nodes)
    (No features)



## Part 1: Training model

We train a model to predict edges between papers.

In [6]:
model = dgf.learning.train_link_model(
    graph=graph,
    schema=schema,
    target_edgeset="edges",
    # Make the training faster for this demo.
    num_sampling_hops=1,
)

Using gpu JAX backend
Graph input schema:
Node Sets:
  nodes:
    | Feature   | Format     | Semantic   | Shape   | Num cat. vals   |
    |-----------|------------|------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID | None    | None            |
    | feat      | FLOAT_32   | EMBEDDING  | (128,)  | None            |
    | year      | INTEGER_64 | NUMERICAL  | None    | None            |


Edge Sets:
  edges: (Source: nodes, Target: nodes)
    (No features)

Preparing dataset
Num. training seed edges: 1158243, Num. validation seed edges: 8000
Create graph sampler
Compute feature statistics
  Source stats:
GraphFeatureStatistics:
  Node Sets (1):
    'nodes':
      '#id': count=19584, min=nan, max=nan
      'feat': count=19584, min=nan, max=nan
      'year': count=19584, min=1992.0000, max=2020.0000, quantiles=(100)[1996.0000, 2007.0000, 2009.0000, ..., 2020.0000, 2020.0000, 2020.0000]

  Target stats:
GraphFeatureStatistics:
  Node Sets (1):
    'nodes':


[Warning] No normalizer created for node set 'nodes', feature '#id'.
[Warning] No normalizer created for node set 'nodes', feature '#id'.


Compute graph statistics for padding
  positive source padding: Node Sets:
  nodes: 228 nodes

Edge Sets:
  edges: 219 edges
  positive target padding: Node Sets:
  nodes: 263 nodes

Edge Sets:
  edges: 257 edges
  negative target padding: Node Sets:
  nodes: 873 nodes

Edge Sets:
  edges: 808 edges
Preparing dataset finished in 2.50 seconds
Source normalizer:
Graph Normalizer:

Node Sets:
  nodes:
    - feat: IdentityNormalizer
    - year: SoftQuantileNormalizer

Edge Sets:
  edges: (Source: nodes, Target: nodes)
    (No normalizers)

Target normalizer:
Graph Normalizer:

Node Sets:
  nodes:
    - feat: IdentityNormalizer
    - year: SoftQuantileNormalizer

Edge Sets:
  edges: (Source: nodes, Target: nodes)
    (No normalizers)

Source normalized graph schema:
Node Sets:
  nodes:
    | Feature            | Format   | Semantic   | Shape   | Num cat. vals   |
    |--------------------|----------|------------|---------|-----------------|
    | feat               | FLOAT_32 | EMBEDDING  |

Caching validation dataset: 1000it [00:06, 151.96it/s]

Caching validation dataset finished in 6.58 seconds
Number of cache validation batches: 1000
Training model
Generate first batch to initialize model


...

step:8000 train-auc:0.9667 train-hit_at_1:0.8325 train-hit_at_5:0.9962 train-loss:0.0573 train-mrr:0.9038 valid-auc:0.9674 valid-hit_at_1:0.8360 valid-hit_at_5:0.9945 valid-loss:0.0571 valid-mrr:0.9063


Training:  90%|█████████ | 9015/10000 [02:36<00:24, 40.87it/s, step=9000, train-auc=0.9700, train-hit_at_1=0.8325, train-hit_at_5=0.9975, train-loss=0.0520, train-mrr=0.9072, valid-auc=0.9674, valid-hit_at_1=0.8360, valid-hit_at_5=0.9945, valid-loss=0.0571, valid-mrr=0.9063]

step:9000 train-auc:0.9700 train-hit_at_1:0.8325 train-hit_at_5:0.9975 train-loss:0.0520 train-mrr:0.9072 valid-auc:0.9692 valid-hit_at_1:0.8433 valid-hit_at_5:0.9949 valid-loss:0.0553 valid-mrr:0.9106


Training: 100%|██████████| 10000/10000 [02:46<00:00, 60.02it/s, step=9900, train-auc=0.9709, train-hit_at_1=0.8350, train-hit_at_5=0.9962, train-loss=0.0533, train-mrr=0.9093, valid-auc=0.9692, valid-hit_at_1=0.8433, valid-hit_at_5=0.9949, valid-loss=0.0553, valid-mrr=0.9106]


step:10000 train-auc:0.9709 train-hit_at_1:0.8350 train-hit_at_5:0.9962 train-loss:0.0533 train-mrr:0.9093 valid-auc:0.9693 valid-hit_at_1:0.8431 valid-hit_at_5:0.9954 valid-loss:0.0549 valid-mrr:0.9107
Final metrics: {'step': '9900', 'train-auc': '0.9709', 'train-hit_at_1': '0.8350', 'train-hit_at_5': '0.9962', 'train-loss': '0.0533', 'train-mrr': '0.9093', 'valid-auc': '0.9692', 'valid-hit_at_1': '0.8433', 'valid-hit_at_5': '0.9949', 'valid-loss': '0.0553', 'valid-mrr': '0.9106'}
Training model finished in 189.42 seconds


Once trained, it is important to look at the model:

In [7]:
model.describe()

After training, a model is generally evaluated on a test dataset/graph.

Note: We don't have a test graph, so we use our training dataset here. In a real
pipeline, evaluating a model on a training dataset make little sense.

In [8]:
model.evaluate(graph)

Evaluating model on 10000 edges


Inference:   0%|          | 0/11250 [00:00<?, ?it/s]

...Tracing inference model


Inference:   0%|          | 1/11250 [00:05<17:30:28,  5.60s/it]

...Tracing inference model


Inference: 100%|██████████| 11250/11250 [00:37<00:00, 298.96it/s]


Evaluation(loss=None, accuracy=None, rmse=None, r2=None, num_examples=10000, num_examples_weighted=None, mrr=0.9106455158730136, auc=0.9692, hit_at={1: 0.8428, 5: 0.9949}, user_metrics={})

We can make preditions for individual nodes:

In [15]:
# Predict the probability of each label class for the nodes 0 and 1.
predictions = model.predict(graph, source_node_idxs=[0], target_node_idxs=[1])
print()
print("Source node id:", graph.node_sets["nodes"].features["#id"][0])
print("Target node id:", graph.node_sets["nodes"].features["#id"][1])
print("Probability of an edge between the two:", predictions)

Inference: 100%|██████████| 1/1 [00:00<00:00, 358.79it/s]


Source node id: b'n0'
Target node id: b'n1'
Probability of an edge between the two: [0.15650684]


You can make predictions for groups of nodes (more efficient):

In [17]:
predictions = model.predict(
    graph, source_node_idxs=[0], target_node_idxs=[1, 2, 3, 4, 5]
)
predictions

Inference: 100%|██████████| 1/1 [00:00<00:00, 284.21it/s]


array([0.14819287, 0.5756987 , 0.18221028, 0.35714212, 0.06918042],
      dtype=float32)

## Part 2: Decomposable model embedding a.k.a. Two tower model.

Models trained with `train_link_model` are decomposable by default. You can
compute the embeddings with `predict_embedding`.

In [21]:
source_embedding = model.predict_embedding(
    graph, node_idxs=[0], encoder="source"
)
target_embedding = model.predict_embedding(
    graph, node_idxs=[1, 2, 3, 4, 5], encoder="target"
)
print()
print("source_embedding:", source_embedding.shape)
print("target_embedding:", target_embedding.shape)

Embedding (target): 100%|██████████| 1/1 [00:00<00:00, 535.47it/s]


source_embedding: (1, 128)
target_embedding: (5, 128)


With the embedding, we can re-compute the probabilities predicted by `predict`.

**Note:** GNN inference is stocastic. The values will not exactly match.

In [27]:
jax.nn.sigmoid(np.sum(source_embedding * target_embedding, axis=1))

Array([0.14622566, 0.57280886, 0.16041517, 0.39065528, 0.07079427],      dtype=float32)

## Part 3: Indexing embddings for fast retrieval

First, let's compute target embeddings and index them using a vector database
library. We will use [Faiss](https://faiss.ai/index.html). See
go/vector-db-chooser for google internal solutions.

In [30]:
# Compute all the embeddings.
num_target_nodes = graph.node_sets["nodes"].num_nodes
target_embeddings = model.predict_embedding(
    graph, node_idxs=range(num_target_nodes), encoder="target"
)

Embedding (target): 100%|██████████| 21168/21168 [00:30<00:00, 693.56it/s]


Now, let's index them.

**Note:** As a temporary solution, index values with a simple (but inefficient)
numpy based solution.

In [37]:
# As a temporary solution, index values with a simple (but inefficient) numpy based solution.
# TODO: Update code when Faiss is availabe.
def index_embeddings(embeddings: np.ndarray):

  def sigmoid(x):
    return 1 / (1 + np.exp(-x))

  def find_closest_nodes(query: np.ndarray, n_neighbors: int = 5) -> np.ndarray:
    # Compute similarities
    similarities = np.dot(embeddings, query.T).squeeze()
    # Get top k indices (highest similarity)
    top_idxs = np.argsort(similarities)[-n_neighbors:][::-1]
    # Convert similarity to probabilities
    top_probas = sigmoid(similarities[top_idxs])
    return top_idxs, top_probas

  return find_closest_nodes


# Query the index
index = index_embeddings(target_embeddings)

Finally, we can take the `source_embedding` we computed in Part 2 and query the
index to retrieve the most likely connections instantly, without having to run
the model on all pairs.

In [45]:
# Query the index using our source embedding
idxs, probas = index(source_embedding[0])

print("5 most likely connections to query node:")
for idx, proba in zip(idxs, probas):
  id = graph.node_sets["nodes"].features["#id"][idx]
  print(f"\tid={id} idx={idx} proba={proba}")

5 most likely connections to query node:
	id=b'n93487' idx=93487 proba=0.9989976286888123
	id=b'n0' idx=0 proba=0.9989595413208008
	id=b'n30038' idx=30038 proba=0.9983300566673279
	id=b'n44797' idx=44797 proba=0.9965341091156006
	id=b'n93663' idx=93663 proba=0.9956117272377014
